# Data Analysis - Capa GOLD

En este notebook se realiza la transformación de los datos de la capa SILVER para la creación de información analítica.

El objetivo es generar información agregada para el análisis de clientes y sus gastos.

In [1]:
import os
import sys

# Configurar variables de entorno para Hadoop en Windows
HADOOP_HOME = os.environ.get("HADOOP_HOME", "C:\\hadoop")

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["hadoop.home.dir"] = HADOOP_HOME
os.environ["PATH"] = f"{os.environ.get('PATH', '')};{HADOOP_HOME}\\bin"

# Configurar Python para Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Configurar memoria antes de iniciar Spark
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 4g pyspark-shell"
)

In [2]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    min,
    max,
    round,
    countDistinct
)

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [3]:
# Crear sesión de Spark
spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin_Gold")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

# Comprobar la memoria disponible para Java
memoria_java_gb = (
    spark.sparkContext._jvm.java.lang.Runtime
    .getRuntime()
    .maxMemory()
    / (1024 ** 3)
)

print("Spark inicializado")
print(f"Memoria máxima de Java: {memoria_java_gb:.1f} GB")

Spark inicializado
Memoria máxima de Java: 4.0 GB


## Cargar los conjuntos de datos

In [4]:
# Cargar datasets

INPUT_PATH = "../data/processed/silver"
OUTPUT_PATH = "../data/processed/gold"

import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

users_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/users.parquet")
)

cards_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/cards.parquet")
)

transactions_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/transactions.parquet")
)

print("Datos cargados")

Datos cargados


In [5]:
# Cargar cuentas y obligaciones

from pyspark.sql import functions as F
from pyspark.sql.window import Window

accounts_spark = spark.read.parquet(
    f"{INPUT_PATH}/accounts.parquet"
)

obligations_spark = spark.read.parquet(
    f"{INPUT_PATH}/obligations_payments.parquet"
)


# Separar convenciones de transacciones

legacy_transactions = transactions_spark.filter(
    F.col("amount_convention") == "legacy_unchanged"
)

movements_spark = transactions_spark.filter(
    F.col("amount_convention") == "signed_cashflow"
)

## Explorar la información

In [6]:
# Mostrar primeras filas

users_spark.show(5)
cards_spark.show(5)
transactions_spark.show(5)

+---+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+--------------------+--------------------+----------------+----------------+-------+-----------+--------+
| id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|           full_name|               email|            city|           state|country|postal_code|currency|
+---+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+--------------------+--------------------+----------------+----------------+-------+-----------+--------+
|  0|       33.0|            69|      1986|          3|  male|Calle Reforma 441...| 22.1565|-100.9855|         526266.0|    1073034.0|  651582.0|         763

## Definir las validaciones de calidad

Se revisan los tipos de datos, identificadores, valores duplicados,
rangos y referencias entre los conjuntos de datos.

Los identificadores inválidos y duplicados tienen tolerancia cero.
Las fechas faltantes de transacciones tienen una tolerancia del 6 %.

Si alguna regla no se cumple, se detiene el procesamiento.

In [6]:
# Importar librerías

from pyspark.sql import functions as F
from pyspark.sql import types as T


# Definir tipos permitidos

TIPOS_ENTEROS = (
    T.ByteType,
    T.ShortType,
    T.IntegerType,
    T.LongType
)

TIPOS_NUMERICOS = TIPOS_ENTEROS + (
    T.FloatType,
    T.DoubleType,
    T.DecimalType
)

TIPOS_TEXTO = (T.StringType,)
TIPOS_BOOLEANOS = (T.BooleanType,)
TIPOS_FECHA = (T.TimestampType, T.DateType)

MAX_PCT_FECHAS_NULAS = 6.0


class ErrorCalidadDatos(ValueError):
    pass


# Validar valores disponibles

def fuera_de_rango(columna, minimo, maximo=None):
    valor = F.col(columna)
    error = valor < minimo

    if maximo is not None:
        error = error | (valor > maximo)

    return valor.isNotNull() & error


def tiene_decimales(columna):
    valor = F.col(columna)

    return (
        valor.isNotNull()
        & (valor != F.floor(valor))
    )


# Validar calidad de una tabla

def validar_tabla(nombre, df, tipos, reglas, clave):
    esquema = {
        campo.name: campo.dataType
        for campo in df.schema.fields
    }

    errores = []

    for columna, tipos_permitidos in tipos.items():
        if columna not in esquema:
            errores.append(f"Falta la columna {columna}")

        elif not isinstance(esquema[columna], tipos_permitidos):
            errores.append(
                f"{columna}: tipo incorrecto "
                f"({esquema[columna].simpleString()})"
            )

    if errores:
        raise ErrorCalidadDatos(
            f"{nombre}: esquema rechazado.\n"
            + "\n".join(errores)
        )

    reglas = list(reglas)

    reglas.append((
        f"{clave}: nulo o negativo",
        F.col(clave).isNull() | (F.col(clave) < 0),
        0.0
    ))

    # Revisar valores numéricos no finitos
    for columna, tipo in esquema.items():
        if isinstance(tipo, TIPOS_NUMERICOS):
            valor = F.col(columna).cast("double")

            reglas.append((
                f"{columna}: NaN o infinito",
                F.isnan(valor)
                | (F.abs(valor) == F.lit(float("inf"))),
                0.0
            ))

    expresiones = [
        F.count(F.lit(1)).alias("total"),
        F.count(clave).alias("ids_presentes"),
        F.countDistinct(clave).alias("ids_distintos")
    ]

    for indice, (_, condicion, _) in enumerate(reglas):
        expresiones.append(
            F.sum(
                F.when(
                    F.coalesce(condicion, F.lit(False)),
                    1
                ).otherwise(0)
            ).alias(f"error_{indice}")
        )

    resultado = df.agg(*expresiones).first()
    total = resultado["total"]

    if total == 0:
        raise ErrorCalidadDatos(
            f"{nombre}: el conjunto de datos está vacío."
        )

    mediciones = [
        (
            nombre_regla,
            int(resultado[f"error_{indice}"] or 0),
            tolerancia
        )
        for indice, (nombre_regla, _, tolerancia)
        in enumerate(reglas)
    ]

    duplicados = (
        resultado["ids_presentes"]
        - resultado["ids_distintos"]
    )

    mediciones.append((
        f"{clave}: duplicados adicionales",
        int(duplicados),
        0.0
    ))

    fallos = []

    print(f"\nCalidad de {nombre}: {total} registros")

    for regla, cantidad, tolerancia in mediciones:
        porcentaje = cantidad * 100.0 / total
        cumple = porcentaje <= tolerancia
        estado = "OK" if cumple else "ERROR"

        print(
            f"[{estado}] {regla}: "
            f"{cantidad}/{total} ({porcentaje:.2f} %), "
            f"máximo {tolerancia:.2f} %"
        )

        if not cumple:
            fallos.append(regla)

    if fallos:
        raise ErrorCalidadDatos(
            f"{nombre}: calidad rechazada.\n"
            + "\n".join(fallos)
        )

    return total

## Validar los datos de SILVER

Se valida la información antes de crear los resúmenes.

Los datos opcionales del cliente pueden permanecer como valores nulos.
Los identificadores y los importes de tarjetas y transacciones
deben estar disponibles.

También se verifica que las tarjetas y transacciones correspondan
a clientes existentes.

In [7]:
# Definir validaciones de entrada

def validar_entrada_gold(users, cards, transactions):
    tipos_users = {
        "id": TIPOS_ENTEROS,
        "current_age": TIPOS_NUMERICOS,
        "retirement_age": TIPOS_NUMERICOS,
        "gender": TIPOS_TEXTO,
        "yearly_income": TIPOS_NUMERICOS,
        "per_capita_income": TIPOS_NUMERICOS,
        "total_debt": TIPOS_NUMERICOS,
        "credit_score": TIPOS_NUMERICOS,
        "num_credit_cards": TIPOS_NUMERICOS
    }

    reglas_users = []

    for columna in [
        "current_age",
        "retirement_age",
        "credit_score",
        "num_credit_cards"
    ]:
        reglas_users.append((
            f"{columna}: valor con decimales",
            tiene_decimales(columna),
            0.0
        ))

    for columna, minimo, maximo in [
        ("current_age", 18, 120),
        ("retirement_age", 0, 120),
        ("credit_score", 300, 850),
        ("num_credit_cards", 0, None),
        ("yearly_income", 0, None),
        ("per_capita_income", 0, None),
        ("total_debt", 0, None)
    ]:
        reglas_users.append((
            f"{columna}: fuera de rango",
            fuera_de_rango(columna, minimo, maximo),
            0.0
        ))

    reglas_users.append((
        "gender: vacío o mayor de 10 caracteres",
        F.col("gender").isNotNull()
        & (
            (F.length(F.trim(F.col("gender"))) == 0)
            | (F.length(F.col("gender")) > 10)
        ),
        0.0
    ))

    total_users = validar_tabla(
        "users_silver",
        users,
        tipos_users,
        reglas_users,
        "id"
    )

    validar_tabla(
        "cards_silver",
        cards,
        {
            "id": TIPOS_ENTEROS,
            "client_id": TIPOS_ENTEROS,
            "card_brand": TIPOS_TEXTO,
            "card_type": TIPOS_TEXTO,
            "credit_limit": TIPOS_NUMERICOS,
            "credit_limit_imputed": TIPOS_BOOLEANOS
        },
        [
            (
                "client_id: nulo o negativo",
                F.col("client_id").isNull()
                | (F.col("client_id") < 0),
                0.0
            ),
            (
                "credit_limit: nulo o negativo",
                F.col("credit_limit").isNull()
                | (F.col("credit_limit") < 0),
                0.0
            ),
            (
                "credit_limit_imputed: nulo",
                F.col("credit_limit_imputed").isNull(),
                0.0
            )
        ],
        "id"
    )

    validar_tabla(
        "transactions_silver",
        transactions,
        {
            "id": TIPOS_ENTEROS,
            "client_id": TIPOS_ENTEROS,
            "amount": TIPOS_NUMERICOS,
            "date": TIPOS_FECHA,
            "merchant_city": TIPOS_TEXTO
        },
        [
            (
                "client_id: nulo o negativo",
                F.col("client_id").isNull()
                | (F.col("client_id") < 0),
                0.0
            ),
            (
                "amount: nulo",
                F.col("amount").isNull(),
                0.0
            ),
            (
                "date: nula",
                F.col("date").isNull(),
                MAX_PCT_FECHAS_NULAS
            )
        ],
        "id"
    )

    # Validar referencias a clientes
    clientes = users.select(
        F.col("id").alias("client_id")
    )

    for nombre, datos in [
        ("cards_silver", cards),
        ("transactions_silver", transactions)
    ]:
        sin_cliente = (
            datos.select("client_id")
            .join(clientes, "client_id", "left_anti")
            .count()
        )

        if sin_cliente:
            raise ErrorCalidadDatos(
                f"{nombre}: {sin_cliente} registros "
                "sin un cliente existente."
            )

    return total_users


# Validar entrada

total_users = validar_entrada_gold(
    users_spark,
    cards_spark,
    transactions_spark
)

print("Datos de entrada aprobados")


Calidad de users_silver: 2038 registros
[OK] current_age: valor con decimales: 0/2038 (0.00 %), máximo 0.00 %
[OK] retirement_age: valor con decimales: 0/2038 (0.00 %), máximo 0.00 %
[OK] credit_score: valor con decimales: 0/2038 (0.00 %), máximo 0.00 %
[OK] num_credit_cards: valor con decimales: 0/2038 (0.00 %), máximo 0.00 %
[OK] current_age: fuera de rango: 0/2038 (0.00 %), máximo 0.00 %
[OK] retirement_age: fuera de rango: 0/2038 (0.00 %), máximo 0.00 %
[OK] credit_score: fuera de rango: 0/2038 (0.00 %), máximo 0.00 %
[OK] num_credit_cards: fuera de rango: 0/2038 (0.00 %), máximo 0.00 %
[OK] yearly_income: fuera de rango: 0/2038 (0.00 %), máximo 0.00 %
[OK] per_capita_income: fuera de rango: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_debt: fuera de rango: 0/2038 (0.00 %), máximo 0.00 %
[OK] gender: vacío o mayor de 10 caracteres: 0/2038 (0.00 %), máximo 0.00 %
[OK] id: nulo o negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] id: NaN o infinito: 0/2038 (0.00 %), máximo 0.00 %
[OK] curre

## Crear información de tarjetas por cliente

In [8]:
# Crear resumen de tarjetas

cards_gold = (
    cards_spark
    .groupBy("client_id")
    .agg(
        count("*").alias("total_cards"),
        count("id").alias("total_cards_with_id"),
        countDistinct("card_brand").alias("total_card_brands"),
        countDistinct("card_type").alias("total_card_types"),
        round(avg("credit_limit"), 2).alias("average_credit_limit"),
        round(sum("credit_limit"), 2).alias("total_credit_limit"),
        sum(
            col("credit_limit_imputed").cast("int")
        ).alias("total_credit_limits_imputed")
    )
    .withColumn(
        "total_cards_without_id",
        col("total_cards") - col("total_cards_with_id")
    )
    .withColumn(
        "has_imputed_credit_limit",
        col("total_credit_limits_imputed") > 0
    )
)

## Crear información de transacciones anteriores por cliente

Se conserva el resumen de las transacciones con la convención anterior.

Los movimientos de cuenta se procesan por separado para evitar
mezclar gastos con ingresos, pagos y transferencias.

In [9]:
# Crear resumen de transacciones anteriores

transactions_gold = (
    legacy_transactions
    .groupBy("client_id")
    .agg(
        F.count("id").alias("total_transactions"),
        F.round(
            F.sum("amount"), 2
        ).alias("total_spent"),
        F.round(
            F.avg("amount"), 2
        ).alias("average_transaction"),
        F.round(
            F.min("amount"), 2
        ).alias("minimum_transaction"),
        F.round(
            F.max("amount"), 2
        ).alias("maximum_transaction"),
        F.countDistinct(
            "merchant_city"
        ).alias("total_merchant_cities")
    )
)

## Crear información financiera por cliente

In [10]:
# Crear información financiera

users_gold = users_spark.select(
    "id",
    "full_name",
    "email",
    "city",
    "state",
    "country",
    "postal_code",
    "current_age",
    "retirement_age",
    "gender",
    "yearly_income",
    "per_capita_income",
    "total_debt",
    "credit_score",
    "num_credit_cards",
    F.col("currency").alias("profile_currency")
)

## Unir la información de clientes

In [11]:
# Unir información

customer_gold = (
    users_gold
    .join(
        cards_gold,
        users_gold.id == cards_gold.client_id,
        "left"
    )
    .drop(cards_gold.client_id)
    .join(
        transactions_gold,
        users_gold.id == transactions_gold.client_id,
        "left"
    )
    .drop(transactions_gold.client_id)
)

## Identificar información disponible

Se identifica si el cliente tiene registros de tarjetas y transacciones en los datos procesados.

La ausencia de registros no implica que el cliente no tenga tarjetas o que su gasto sea cero.

In [12]:
# Identificar información disponible

customer_gold = (
    customer_gold
    .withColumn(
        "has_card_records",
        col("total_cards").isNotNull()
    )
    .withColumn(
        "has_transaction_records",
        col("total_transactions").isNotNull()
    )
)

## Crear métricas financieras

In [13]:
# Crear métricas financieras

customer_gold = (
    customer_gold
    .withColumn(
        "spending_to_income_ratio",
        F.when(
            F.col("yearly_income").isNotNull()
            & (F.col("yearly_income") != 0),
            F.round(
                F.col("total_spent") / F.col("yearly_income"),
                4
            )
        )
    )
    .withColumn(
        "debt_to_income_ratio",
        F.when(
            F.col("yearly_income").isNotNull()
            & (F.col("yearly_income") != 0),
            F.round(
                F.col("total_debt") / F.col("yearly_income"),
                4
            )
        )
    )
)

## Crear el historial mensual de cuentas

Se construye una fila por cuenta y mes completo disponible.

Los ingresos, gastos, pagos de deuda y transferencias se calculan
por separado.

Los meses sin movimientos conservan el saldo anterior. Los valores
cero corresponden a meses cubiertos por el historial de la cuenta.

In [14]:
# Definir meses completos disponibles

accounts_calendar = (
    accounts_spark
    .withColumn(
        "first_month",
        F.when(
            F.dayofmonth("opened_at") == 1,
            F.trunc(F.to_date("opened_at"), "month")
        ).otherwise(
            F.add_months(
                F.trunc(F.to_date("opened_at"), "month"),
                1
            )
        )
    )
    .withColumn(
        "last_month",
        F.when(
            F.to_date("balance_as_of")
            == F.last_day("balance_as_of"),
            F.trunc(F.to_date("balance_as_of"), "month")
        ).otherwise(
            F.add_months(
                F.trunc(F.to_date("balance_as_of"), "month"),
                -1
            )
        )
    )
    .filter(F.col("first_month") <= F.col("last_month"))
    .withColumn(
        "month",
        F.explode(
            F.sequence(
                "first_month",
                "last_month",
                F.expr("interval 1 month")
            )
        )
    )
    .select(
        "account_id",
        "client_id",
        "currency",
        "opening_balance",
        "month"
    )
)


# Preparar movimientos mensuales

movements_monthly = movements_spark.withColumn(
    "month",
    F.trunc(F.to_date("date"), "month")
)


# Agregar flujos por cuenta

def sumar_movimiento(tipo, nombre, signo=1):
    return F.sum(
        F.when(
            F.col("movement_type") == tipo,
            F.col("amount") * signo
        ).otherwise(0)
    ).alias(nombre)


account_flows = (
    movements_monthly
    .groupBy("account_id", "month")
    .agg(
        F.count("*").alias("monthly_transactions"),
        sumar_movimiento("income", "monthly_income"),
        sumar_movimiento("expense", "monthly_expenses", -1),
        sumar_movimiento(
            "debt_payment",
            "monthly_debt_payments",
            -1
        ),
        sumar_movimiento("transfer_in", "transfers_in"),
        sumar_movimiento("transfer_out", "transfers_out", -1),
        F.sum("amount").alias("monthly_net_flow"),
        F.max(
            F.struct("date", "id", "balance_after")
        ).alias("last_movement")
    )
    .withColumn(
        "observed_balance",
        F.col("last_movement.balance_after")
    )
    .drop("last_movement")
)


# Completar meses cubiertos

columnas_flujo = [
    "monthly_transactions",
    "monthly_income",
    "monthly_expenses",
    "monthly_debt_payments",
    "transfers_in",
    "transfers_out",
    "monthly_net_flow"
]

account_monthly = (
    accounts_calendar
    .join(
        account_flows,
        ["account_id", "month"],
        "left"
    )
    .fillna(0, subset=columnas_flujo)
)


# Recuperar saldo al cierre de cada mes

ventana_cuenta = (
    Window.partitionBy("account_id")
    .orderBy("month")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

account_monthly = account_monthly.withColumn(
    "closing_balance",
    F.coalesce(
        F.last(
            "observed_balance",
            ignorenulls=True
        ).over(ventana_cuenta),
        F.col("opening_balance")
    )
)


# Crear resumen mensual por cliente y moneda

monthly_gold = (
    account_monthly
    .groupBy("client_id", "currency", "month")
    .agg(
        *[
            F.sum(columna).alias(columna)
            for columna in columnas_flujo
        ],
        F.sum("closing_balance").alias("closing_balance"),
        F.countDistinct("account_id").alias("total_accounts")
    )
    .withColumn(
        "monthly_savings",
        F.col("monthly_income")
        - F.col("monthly_expenses")
        - F.col("monthly_debt_payments")
    )
    .withColumn(
        "savings_rate",
        F.when(
            F.col("monthly_income") > 0,
            F.round(
                F.col("monthly_savings")
                / F.col("monthly_income"),
                4
            )
        )
    )
)

## Calcular obligaciones pendientes por mes

Se reconstruye el importe pendiente al cierre de cada mes.

Solo se consideran las cuotas vencidas hasta ese momento y los
pagos que ya habían ocurrido.

No se utilizan los estados finales de las obligaciones como
información de meses anteriores.

In [15]:
# Preparar cortes mensuales

cortes = (
    monthly_gold
    .select("client_id", "currency", "month")
    .withColumn(
        "cutoff_date",
        F.last_day("month")
    )
)


# Relacionar obligaciones con cada corte

obligaciones_historicas = (
    cortes
    .join(
        obligations_spark.select(
            "client_id",
            "currency",
            "obligation_id",
            "due_date",
            "payment_date",
            "amount_due",
            "amount_paid"
        ),
        ["client_id", "currency"],
        "left"
    )
    .withColumn(
        "due_at_cutoff",
        F.when(
            F.col("due_date") <= F.col("cutoff_date"),
            F.col("amount_due")
        ).otherwise(0)
    )
    .withColumn(
        "paid_at_cutoff",
        F.when(
            (F.col("due_date") <= F.col("cutoff_date"))
            & (F.col("payment_date") <= F.col("cutoff_date")),
            F.col("amount_paid")
        ).otherwise(0)
    )
    .withColumn(
        "outstanding_at_cutoff",
        F.col("due_at_cutoff") - F.col("paid_at_cutoff")
    )
    .withColumn(
        "days_past_due",
        F.when(
            F.col("outstanding_at_cutoff") > 0,
            F.greatest(
                F.datediff("cutoff_date", "due_date"),
                F.lit(0)
            )
        ).otherwise(0)
    )
)


# Agregar obligaciones pendientes

obligations_monthly = (
    obligaciones_historicas
    .groupBy("client_id", "currency", "month")
    .agg(
        F.sum(
            "outstanding_at_cutoff"
        ).alias("outstanding_due"),
        F.max(
            "days_past_due"
        ).alias("max_days_past_due"),
        F.sum(
            F.when(
                (F.col("outstanding_at_cutoff") > 0)
                & (F.col("days_past_due") >= 30),
                1
            ).otherwise(0)
        ).alias("installments_30_days_past_due")
    )
)

monthly_gold = monthly_gold.join(
    obligations_monthly,
    ["client_id", "currency", "month"],
    "left"
)

## Crear variables de historial financiero

Se calculan rezagos y estadísticas de los últimos meses.

Cada fila utiliza únicamente información disponible hasta
el cierre de su mes.

También se incorpora el último resumen mensual al perfil del cliente.

In [16]:
# Definir ventanas temporales

ventana_cliente = Window.partitionBy(
    "client_id",
    "currency"
).orderBy("month")

ventana_tres_meses = ventana_cliente.rowsBetween(-2, 0)

ventana_historial = ventana_cliente.rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)


# Crear variables mensuales

monthly_gold = (
    monthly_gold
    .withColumn(
        "history_months",
        F.count("*").over(ventana_historial)
    )
    .withColumn(
        "income_lag_1",
        F.lag("monthly_income", 1).over(ventana_cliente)
    )
    .withColumn(
        "income_lag_3",
        F.lag("monthly_income", 3).over(ventana_cliente)
    )
    .withColumn(
        "income_mean_3",
        F.avg("monthly_income").over(ventana_tres_meses)
    )
    .withColumn(
        "income_std_3",
        F.stddev_samp("monthly_income").over(
            ventana_tres_meses
        )
    )
    .withColumn(
        "expenses_mean_3",
        F.avg("monthly_expenses").over(ventana_tres_meses)
    )
    .withColumn(
        "negative_balance_months_3",
        F.sum(
            F.when(
                F.col("closing_balance") < 0,
                1
            ).otherwise(0)
        ).over(ventana_tres_meses)
    )
)


# Verificar continuidad mensual

revision_meses = monthly_gold.withColumn(
    "previous_month",
    F.lag("month").over(ventana_cliente)
)

if (
    revision_meses
    .filter(
        F.col("previous_month").isNotNull()
        & (
            F.col("month")
            != F.add_months("previous_month", 1)
        )
    )
    .limit(1)
    .count()
):
    raise ValueError(
        "El historial mensual contiene períodos discontinuos."
    )


# Seleccionar último mes de cada cliente

ventana_ultimo = Window.partitionBy(
    "client_id",
    "currency"
).orderBy(F.col("month").desc())

latest_monthly = (
    monthly_gold
    .withColumn(
        "position",
        F.row_number().over(ventana_ultimo)
    )
    .filter(F.col("position") == 1)
    .drop("position")
)


# Evitar mezclar monedas en el perfil

if (
    latest_monthly.groupBy("client_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    raise ValueError(
        "Hay clientes con varias monedas. "
        "El perfil debe conservar un resumen por moneda."
    )


# Preparar último resumen

latest_summary = latest_monthly.select(
    F.col("client_id").alias("id"),
    "currency",
    F.last_day("month").alias("financial_cutoff_date"),
    F.col("closing_balance").alias("account_balance"),
    "monthly_income",
    "monthly_expenses",
    "monthly_debt_payments",
    "monthly_savings",
    "monthly_transactions",
    "transfers_in",
    "transfers_out",
    "outstanding_due",
    "max_days_past_due",
    "installments_30_days_past_due",
    "income_mean_3",
    "income_std_3",
    "expenses_mean_3",
    "history_months"
)

customer_gold = customer_gold.join(
    latest_summary,
    "id",
    "left"
)

monthly_gold.orderBy(
    "client_id",
    "month"
).show(10, truncate=False)

+---------+--------+----------+--------------------+--------------+----------------+---------------------+------------+-------------+----------------+---------------+--------------+---------------+------------+---------------+-----------------+-----------------------------+--------------+------------+------------+-------------+------------------+---------------+-------------------------+
|client_id|currency|month     |monthly_transactions|monthly_income|monthly_expenses|monthly_debt_payments|transfers_in|transfers_out|monthly_net_flow|closing_balance|total_accounts|monthly_savings|savings_rate|outstanding_due|max_days_past_due|installments_30_days_past_due|history_months|income_lag_1|income_lag_3|income_mean_3|income_std_3      |expenses_mean_3|negative_balance_months_3|
+---------+--------+----------+--------------------+--------------+----------------+---------------------+------------+-------------+----------------+---------------+--------------+---------------+------------+--------

## Mostrar información final

In [17]:
# Mostrar información final

customer_gold.show(10)

+---+--------------------+--------------------+----------------+----------------+-------+-----------+-----------+--------------+------+-------------+-----------------+----------+------------+----------------+----------------+-----------+-------------------+-----------------+----------------+--------------------+------------------+---------------------------+----------------------+------------------------+------------------+-----------+-------------------+-------------------+-------------------+---------------------+----------------+-----------------------+------------------------+--------------------+--------+---------------------+---------------+--------------+----------------+---------------------+---------------+--------------------+------------+-------------+---------------+-----------------+-----------------------------+-------------+------------------+---------------+--------------+
| id|           full_name|               email|            city|           state|country|postal_co

## Completar las variables para modelaje

Se incorpora el número del mes al historial mensual.

Se agregan al resumen del cliente los ingresos de meses anteriores
y la cantidad de meses con saldo negativo.

Las variables corresponden al último mes disponible de cada cliente.
Los valores ausentes se conservan como información no disponible.

In [18]:
# Incorporar el mes como variable predictora

monthly_gold = monthly_gold.withColumn(
    "month_number",
    F.month("month")
)


# Seleccionar las variables del último mes disponible

features_cliente = (
    monthly_gold
    .withColumn(
        "position",
        F.row_number().over(ventana_ultimo)
    )
    .filter(F.col("position") == 1)
    .select(
        F.col("client_id").alias("id"),
        "currency",
        F.last_day("month").alias("financial_cutoff_date"),
        "income_lag_1",
        "income_lag_3",
        "negative_balance_months_3",
        "month_number"
    )
)


# Incorporar las variables al resumen del cliente
# Eliminar previamente estas columnas permite repetir la celda

columnas_features = [
    "income_lag_1",
    "income_lag_3",
    "negative_balance_months_3",
    "month_number"
]

customer_gold = (
    customer_gold
    .drop(*columnas_features)
    .join(
        features_cliente,
        ["id", "currency", "financial_cutoff_date"],
        "left"
    )
)

## Crear información reciente y de salud financiera

Se conservan los últimos tres meses completos de cada cliente.

Se resumen los gastos por categoría y se calculan indicadores
de ingresos, gastos, ahorro, pagos de deuda y cobertura del saldo.

Los indicadores permiten explicar la situación financiera
correspondiente al corte disponible.

In [19]:
# Conservar una sola columna de moneda en el perfil
customer_gold = (
    customer_gold
    .withColumn(
        "currency",
        F.coalesce(
            F.col("currency"),
            F.col("profile_currency")
        )
    )
    .drop("profile_currency")
    .withColumn(
        "has_financial_history",
        F.col("financial_cutoff_date").isNotNull()
    )
)


# Seleccionar los últimos tres meses completos
ventana_reciente = Window.partitionBy(
    "client_id",
    "currency"
).orderBy(F.col("month").desc())

recent_months_gold = (
    monthly_gold
    .withColumn(
        "_position",
        F.row_number().over(ventana_reciente)
    )
    .filter(F.col("_position") <= 3)
    .drop("_position")
)


# Crear información de gastos por categoría y mes
expense_categories_gold = (
    movements_spark
    .filter(F.col("movement_type") == "expense")
    .withColumn(
        "month",
        F.trunc(F.to_date("date"), "month")
    )
    .join(
        monthly_gold.select(
            "client_id",
            "currency",
            "month"
        ),
        ["client_id", "currency", "month"],
        "left_semi"
    )
    .groupBy(
        "client_id",
        "currency",
        "month",
        "category"
    )
    .agg(
        F.count("*").alias("expense_transactions"),
        F.sum(-F.col("amount")).alias("total_expenses")
    )
)


# Resumir los últimos tres meses por categoría
recent_categories_gold = (
    expense_categories_gold
    .join(
        recent_months_gold.select(
            "client_id",
            "currency",
            "month"
        ),
        ["client_id", "currency", "month"],
        "left_semi"
    )
    .groupBy(
        "client_id",
        "currency",
        "category"
    )
    .agg(
        F.sum("expense_transactions").alias(
            "expense_transactions"
        ),
        F.sum("total_expenses").alias("total_expenses")
    )
)


# Crear indicadores para consultar la salud financiera
health_gold = (
    recent_months_gold
    .groupBy("client_id", "currency")
    .agg(
        F.count("*").alias("months_available_3m"),
        F.min("month").alias("period_start"),
        F.last_day(
            F.max("month")
        ).alias("financial_cutoff_date"),
        F.sum("monthly_income").alias("income_3m"),
        F.sum("monthly_expenses").alias("expenses_3m"),
        F.sum(
            "monthly_debt_payments"
        ).alias("debt_payments_3m"),
        F.sum("monthly_savings").alias("savings_3m"),
        F.sum(
            "monthly_transactions"
        ).alias("transactions_3m"),
        F.avg("monthly_income").alias("average_income_3m"),
        F.avg("monthly_expenses").alias("average_expenses_3m")
    )
    .join(
        latest_monthly.select(
            "client_id",
            "currency",
            "closing_balance",
            "outstanding_due",
            "max_days_past_due"
        ),
        ["client_id", "currency"],
        "left"
    )
    .withColumn(
        "savings_rate_3m",
        F.when(
            F.col("income_3m") > 0,
            F.round(
                F.col("savings_3m") / F.col("income_3m"),
                4
            )
        )
    )
    .withColumn(
        "debt_payment_ratio_3m",
        F.when(
            F.col("income_3m") > 0,
            F.round(
                F.col("debt_payments_3m") / F.col("income_3m"),
                4
            )
        )
    )
    .withColumn(
        "expense_coverage_months",
        F.when(
            (
                F.col("expenses_3m")
                + F.col("debt_payments_3m")
            ) > 0,
            F.round(
                F.greatest(
                    F.col("closing_balance"),
                    F.lit(0)
                )
                / (
                    (
                        F.col("expenses_3m")
                        + F.col("debt_payments_3m")
                    )
                    / F.col("months_available_3m")
                ),
                2
            )
        )
    )
    .withColumn(
        "has_overdue_payments",
        (F.col("outstanding_due") > 0)
        & (F.col("max_days_past_due") > 0)
    )
    .withColumn(
        "has_negative_balance",
        F.col("closing_balance") < 0
    )
    .withColumn(
        "has_budget_deficit_3m",
        F.col("savings_3m") < 0
    )
    .withColumn(
        "financial_health_status",
        F.when(
            F.col("months_available_3m") < 3,
            "historial_insuficiente"
        )
        .when(
            F.col("has_overdue_payments")
            | F.col("has_negative_balance")
            | F.col("has_budget_deficit_3m"),
            "requiere_atencion"
        )
        .when(
            F.col("income_3m") <= 0,
            "sin_ingresos"
        )
        .when(
            F.col("savings_3m") > 0,
            "margen_positivo"
        )
        .otherwise("sin_margen")
    )
)


# Incorporar indicadores al resumen del cliente
columnas_salud = [
    "months_available_3m",
    "period_start",
    "income_3m",
    "expenses_3m",
    "debt_payments_3m",
    "savings_3m",
    "transactions_3m",
    "average_income_3m",
    "average_expenses_3m",
    "savings_rate_3m",
    "debt_payment_ratio_3m",
    "expense_coverage_months",
    "has_overdue_payments",
    "has_negative_balance",
    "has_budget_deficit_3m",
    "financial_health_status"
]

customer_gold = (
    customer_gold
    .drop(*columnas_salud)
    .join(
        health_gold.select(
            F.col("client_id").alias("id"),
            "currency",
            "financial_cutoff_date",
            *columnas_salud
        ),
        ["id", "currency", "financial_cutoff_date"],
        "left"
    )
    .withColumn(
        "financial_health_status",
        F.coalesce(
            F.col("financial_health_status"),
            F.lit("historial_insuficiente")
        )
    )
)

print("Historial reciente, categorías e indicadores preparados")

Historial reciente, categorías e indicadores preparados


## Validar el resumen GOLD

Se comprueba que exista una fila por cliente y que los conteos,
indicadores y métricas sean consistentes.

Los valores nulos por ausencia de tarjetas o transacciones
se conservan como información no disponible.

In [20]:
# Definir validaciones de salida

def validar_salida_gold(customer, total_users):
    columnas_tarjetas = [
        "total_cards",
        "total_cards_with_id",
        "total_cards_without_id",
        "total_card_brands",
        "total_card_types",
        "total_credit_limits_imputed",
        "average_credit_limit",
        "total_credit_limit",
        "has_imputed_credit_limit"
    ]

    columnas_transacciones = [
        "total_transactions",
        "total_merchant_cities",
        "total_spent",
        "average_transaction",
        "minimum_transaction",
        "maximum_transaction"
    ]

    conteos = [
        "total_cards",
        "total_cards_with_id",
        "total_cards_without_id",
        "total_card_brands",
        "total_card_types",
        "total_credit_limits_imputed",
        "total_transactions",
        "total_merchant_cities"
    ]

    importes = [
        "yearly_income",
        "total_debt",
        "average_credit_limit",
        "total_credit_limit",
        "total_spent",
        "average_transaction",
        "minimum_transaction",
        "maximum_transaction",
        "spending_to_income_ratio",
        "debt_to_income_ratio"
    ]

    tipos = {
        "id": TIPOS_ENTEROS,
        "has_card_records": TIPOS_BOOLEANOS,
        "has_transaction_records": TIPOS_BOOLEANOS,
        "has_imputed_credit_limit": TIPOS_BOOLEANOS
    }

    tipos.update({
        columna: TIPOS_ENTEROS
        for columna in conteos
    })

    tipos.update({
        columna: TIPOS_NUMERICOS
        for columna in importes
    })

    reglas = [
        (
            f"{columna}: negativo",
            fuera_de_rango(columna, 0),
            0.0
        )
        for columna in conteos + [
            "average_credit_limit",
            "total_credit_limit"
        ]
    ]

    # Validar disponibilidad de información
    for indicador, cantidad, columnas in [
        ("has_card_records", "total_cards", columnas_tarjetas),
        (
            "has_transaction_records",
            "total_transactions",
            columnas_transacciones
        )
    ]:
        reglas.append((
            f"{indicador}: inconsistente",
            ~F.col(indicador).eqNullSafe(
                F.col(cantidad).isNotNull()
            ),
            0.0
        ))

        reglas.append((
            f"{cantidad}: debe ser positivo",
            F.col(indicador)
            & (F.col(cantidad) <= 0),
            0.0
        ))

        for columna in columnas:
            reglas.append((
                f"{columna}: disponibilidad inconsistente",
                ~F.col(columna).isNotNull().eqNullSafe(
                    F.col(indicador)
                ),
                0.0
            ))

    reglas.extend([
        (
            "tarjetas: conteos inconsistentes",
            F.col("has_card_records")
            & ~F.col("total_cards").eqNullSafe(
                F.col("total_cards_with_id")
                + F.col("total_cards_without_id")
            ),
            0.0
        ),
        (
            "tarjetas: indicador de estimaciones inconsistente",
            F.col("has_card_records")
            & ~F.col("has_imputed_credit_limit").eqNullSafe(
                F.col("total_credit_limits_imputed") > 0
            ),
            0.0
        ),
        (
            "transacciones: mínimo, promedio y máximo inconsistentes",
            F.col("has_transaction_records")
            & (
                (
                    F.col("minimum_transaction")
                    > F.col("average_transaction")
                )
                | (
                    F.col("average_transaction")
                    > F.col("maximum_transaction")
                )
            ),
            0.0
        )
    ])

    # Validar conteos relacionados
    for columna, limite in [
        ("total_card_brands", "total_cards"),
        ("total_card_types", "total_cards"),
        ("total_credit_limits_imputed", "total_cards"),
        ("total_merchant_cities", "total_transactions")
    ]:
        reglas.append((
            f"{columna}: mayor que {limite}",
            F.col(columna) > F.col(limite),
            0.0
        ))

    # Validar métricas financieras
    for metrica, numerador in [
        ("spending_to_income_ratio", "total_spent"),
        ("debt_to_income_ratio", "total_debt")
    ]:
        esperado = F.when(
            F.col("yearly_income").isNotNull()
            & (F.col("yearly_income") != 0),
            F.round(
                F.col(numerador) / F.col("yearly_income"),
                4
            )
        )

        reglas.append((
            f"{metrica}: cálculo inconsistente",
            ~F.col(metrica).eqNullSafe(esperado),
            0.0
        ))

    total_gold = validar_tabla(
        "customer_gold",
        customer,
        tipos,
        reglas,
        "id"
    )

    if total_gold != total_users:
        raise ErrorCalidadDatos(
            f"GOLD contiene {total_gold} clientes; "
            f"se esperaban {total_users}."
        )

## Validar y guardar GOLD

Se ejecutan las validaciones antes de escribir los archivos.

Si se detectan errores de calidad, se detiene la celda y no se
inicia la escritura de GOLD.

In [21]:
# Validar resumen final
validar_salida_gold(customer_gold, total_users)


# Validar claves mensuales
if (
    monthly_gold
    .groupBy("client_id", "currency", "month")
    .count()
    .filter("count > 1")
    .limit(1)
    .count()
):
    raise ValueError(
        "Existe más de un resumen por cliente, moneda y mes."
    )


# Validar conciliación de flujos
if monthly_gold.filter(
    F.col("monthly_net_flow") != (
        F.col("monthly_income")
        - F.col("monthly_expenses")
        - F.col("monthly_debt_payments")
        + F.col("transfers_in")
        - F.col("transfers_out")
    )
).limit(1).count():
    raise ValueError(
        "Los flujos mensuales no coinciden."
    )


# Validar que las categorías sumen el gasto mensual
gastos_categorias = (
    expense_categories_gold
    .groupBy("client_id", "currency", "month")
    .agg(
        F.sum("total_expenses").alias("category_expenses")
    )
)

revision_categorias = (
    monthly_gold
    .select(
        "client_id",
        "currency",
        "month",
        "monthly_expenses"
    )
    .join(
        gastos_categorias,
        ["client_id", "currency", "month"],
        "left"
    )
    .fillna(0, subset=["category_expenses"])
)

if revision_categorias.filter(
    F.col("monthly_expenses") != F.col("category_expenses")
).limit(1).count():
    raise ValueError(
        "Los gastos por categoría no coinciden con el total mensual."
    )


# Preparar el resumen mientras se generan las nuevas predicciones
customer_model_gold_base = (
    customer_gold
    .withColumn(
        "feature_month",
        F.lit(None).cast("date")
    )
    .withColumn(
        "prediction_month",
        F.lit(None).cast("date")
    )
    .withColumn(
        "predicted_income",
        F.lit(None).cast("double")
    )
    .withColumn(
        "income_prediction_method",
        F.lit(None).cast("string")
    )
    .withColumn(
        "has_income_prediction",
        F.lit(False)
    )
    .withColumn(
        "prediction_validation_status",
        F.lit("PENDIENTE")
    )
)


# Guardar el resultado de la validación del modelo
def guardar_validacion_modelo(estado, detalle, **conteos):
    columnas = [
        "expected_rows",
        "received_rows",
        "missing_rows",
        "unexpected_rows",
        "duplicate_keys",
        "invalid_rows",
        "xgboost_rows",
        "last_month_rows"
    ]

    esquema = "status string, detail string, " + ", ".join(
        f"{columna} long"
        for columna in columnas
    )

    fila = (
        estado,
        detalle,
        *[conteos.get(columna) for columna in columnas]
    )

    control = (
        spark.createDataFrame([fila], esquema)
        .withColumn(
            "validated_at",
            F.current_timestamp()
        )
    )

    control.write.mode("overwrite").parquet(
        f"{OUTPUT_PATH}/income_model_validation.parquet"
    )

    control.show(truncate=False)


# Guardar GOLD
datasets = {
    "customer_financial_summary": customer_gold,
    "customer_model_summary": customer_model_gold_base,
    "transactions_summary": transactions_gold,
    "cards_summary": cards_gold,
    "customer_monthly_summary": monthly_gold,
    "account_monthly_summary": account_monthly,
    "customer_recent_months": recent_months_gold,
    "customer_expenses_by_category": expense_categories_gold,
    "customer_recent_expenses_by_category": recent_categories_gold,
    "customer_financial_health": health_gold,
    "accounts": accounts_spark,
    "obligations_payments": obligations_spark,
    "movements": movements_spark
}

for nombre, datos in datasets.items():
    datos.write.mode("overwrite").parquet(
        f"{OUTPUT_PATH}/{nombre}.parquet"
    )

esperadas = customer_gold.filter(
    F.col("financial_cutoff_date").isNotNull()
).count()

guardar_validacion_modelo(
    "PENDIENTE",
    "GOLD base generado; falta validar la salida del modelo.",
    expected_rows=esperadas
)

print("GOLD base guardado")


Calidad de customer_gold: 2038 registros
[OK] total_cards: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_cards_with_id: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_cards_without_id: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_card_brands: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_card_types: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_credit_limits_imputed: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_transactions: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_merchant_cities: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] average_credit_limit: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_credit_limit: negativo: 0/2038 (0.00 %), máximo 0.00 %
[OK] has_card_records: inconsistente: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_cards: debe ser positivo: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_cards: disponibilidad inconsistente: 0/2038 (0.00 %), máximo 0.00 %
[OK] total_cards_with_id: disponibilidad inconsistente: 0/2038 (0.00 %), 

## Validar e incorporar las predicciones de ingresos

Se verifica que exista una predicción por cada cliente con historial.

Se comprueban los identificadores, la moneda, el mes utilizado,
el mes estimado y los importes generados.

También se revisan registros faltantes, adicionales y duplicados.

El resultado de la revisión se guarda en income_model_validation.
Las predicciones aprobadas se incorporan al resumen del cliente.

Esta sección se ejecuta después de generar las predicciones
en el notebook 06_modelaje_observabilidad.

In [22]:
# Validar la salida del modelo de ingresos
from pathlib import Path

ruta_predicciones = (
    Path(OUTPUT_PATH) / "income_predictions.parquet"
)

conteos = {}


# Invalidar el resumen anterior antes de revisar una nueva salida
customer_model_gold_base.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/customer_model_summary.parquet"
)

try:
    esperados = (
        customer_gold
        .filter(
            F.col("financial_cutoff_date").isNotNull()
        )
        .select(
            "id",
            "currency",
            F.trunc(
                "financial_cutoff_date",
                "month"
            ).alias("feature_month"),
            F.add_months(
                F.trunc(
                    "financial_cutoff_date",
                    "month"
                ),
                1
            ).alias("prediction_month")
        )
    )

    conteos["expected_rows"] = esperados.count()

    if not ruta_predicciones.exists():
        raise ValueError(
            "Falta income_predictions.parquet. Ejecuta la sección "
            "de predicción de ingresos del notebook 06."
        )

    salida = spark.read.parquet(
        str(ruta_predicciones)
    )

    requeridas = {
        "client_id",
        "currency",
        "feature_month",
        "prediction_month",
        "predicted_income",
        "method"
    }

    if not requeridas.issubset(salida.columns):
        raise ValueError(
            "Faltan columnas en la salida del modelo."
        )

    predicciones_gold = salida.select(
        F.expr(
            "try_cast(client_id as bigint)"
        ).alias("id"),
        F.upper(
            F.trim("currency")
        ).alias("currency"),
        F.to_date(
            "feature_month"
        ).alias("feature_month"),
        F.to_date(
            "prediction_month"
        ).alias("prediction_month"),
        F.expr(
            "try_cast(predicted_income as double)"
        ).alias("predicted_income"),
        F.lower(
            F.trim("method")
        ).alias("income_prediction_method")
    )


    # Validar valores y períodos
    errores = (
        F.col("id").isNull()
        | (F.col("id") < 0)
        | F.col("currency").isNull()
        | (F.col("currency") != "MXN")
        | F.col("feature_month").isNull()
        | (F.dayofmonth("feature_month") != 1)
        | F.col("prediction_month").isNull()
        | (
            F.col("prediction_month")
            != F.add_months("feature_month", 1)
        )
        | F.col("predicted_income").isNull()
        | F.isnan("predicted_income")
        | (
            F.abs(F.col("predicted_income"))
            == F.lit(float("inf"))
        )
        | (F.col("predicted_income") < 0)
        | F.col("income_prediction_method").isNull()
        | ~F.col("income_prediction_method").isin(
            "xgboost",
            "last_month"
        )
    )

    conteos["received_rows"] = predicciones_gold.count()

    conteos["invalid_rows"] = predicciones_gold.filter(
        errores
    ).count()

    conteos["duplicate_keys"] = (
        predicciones_gold
        .groupBy("id", "currency")
        .count()
        .filter("count > 1")
        .count()
    )


    # Comparar cobertura contra el corte actual de GOLD
    claves = [
        "id",
        "currency",
        "feature_month",
        "prediction_month"
    ]

    recibidos = predicciones_gold.select(*claves)

    conteos["missing_rows"] = esperados.join(
        recibidos,
        claves,
        "left_anti"
    ).count()

    conteos["unexpected_rows"] = recibidos.join(
        esperados,
        claves,
        "left_anti"
    ).count()

    conteos["xgboost_rows"] = predicciones_gold.filter(
        F.col("income_prediction_method") == "xgboost"
    ).count()

    conteos["last_month_rows"] = predicciones_gold.filter(
        F.col("income_prediction_method") == "last_month"
    ).count()

    if (
        conteos["expected_rows"] == 0
        or (
            conteos["received_rows"]
            != conteos["expected_rows"]
        )
        or any(
            conteos[c] > 0
            for c in [
                "invalid_rows",
                "duplicate_keys",
                "missing_rows",
                "unexpected_rows"
            ]
        )
    ):
        raise ValueError(
            "La salida del modelo está incompleta o no corresponde "
            "a los clientes, moneda y período de GOLD."
        )


    # Incorporar únicamente predicciones aprobadas
    predicciones_gold = predicciones_gold.withColumn(
        "financial_cutoff_date",
        F.last_day("feature_month")
    )

    customer_model_gold = (
        customer_gold
        .join(
            predicciones_gold,
            ["id", "currency", "financial_cutoff_date"],
            "left"
        )
        .withColumn(
            "has_income_prediction",
            F.col("predicted_income").isNotNull()
        )
        .withColumn(
            "prediction_validation_status",
            F.when(
                F.col("has_income_prediction"),
                "APROBADO"
            ).otherwise("SIN_HISTORIAL")
        )
    )

    customer_model_gold.write.mode("overwrite").parquet(
        f"{OUTPUT_PATH}/customer_model_summary.parquet"
    )

    guardar_validacion_modelo(
        "APROBADO",
        "Cobertura, claves, moneda, períodos e importes aprobados.",
        **conteos
    )

except Exception as error:
    guardar_validacion_modelo(
        "RECHAZADO",
        str(error),
        **conteos
    )
    raise

print("Predicciones validadas e incorporadas a GOLD")

+--------+---------------------------------------------------------+-------------+-------------+------------+---------------+--------------+------------+------------+---------------+--------------------------+
|status  |detail                                                   |expected_rows|received_rows|missing_rows|unexpected_rows|duplicate_keys|invalid_rows|xgboost_rows|last_month_rows|validated_at              |
+--------+---------------------------------------------------------+-------------+-------------+------------+---------------+--------------+------------+------------+---------------+--------------------------+
|APROBADO|Cobertura, claves, moneda, períodos e importes aprobados.|2038         |2038         |0           |0              |0             |0           |2038        |0              |2026-11-03 22:06:01.214058|
+--------+---------------------------------------------------------+-------------+-------------+------------+---------------+--------------+------------+-------

In [23]:
# Finalizar Spark

spark.stop()